In [16]:
import sys
!{sys.executable} -m pip install -q \
    langchain \
    langchain-core \
    langchain-community \
    langchain-text-splitters \
    langchain-openai \
    sentence-transformers \
    faiss-cpu \
    ctransformers==0.2.27

In [17]:
print('OK')

OK


In [18]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.llms import CTransformers
from langchain_core.documents import Document
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

/Users/vikalp/miniconda3/envs/langagent/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/8r/yxvh3xk92g5ds4g46pn27_nr0000gn/T/ipykernel_20746/842955388.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [19]:
from dotenv import load_dotenv
import os
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")

os.environ["OPENAI_API_KEY"] = openai_api_key

In [20]:
#Extract the data fro the PDF 
def load_pdf(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader,
    )
    docs = loader.load()
    return docs


In [21]:
extracted_data = load_pdf("data/")

In [22]:
#Create texts Chunks 

In [23]:
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=20,
    )
    texts_chunks = text_splitter.split_documents(extracted_data)
    return texts_chunks


In [24]:
texts_chunks = text_split(extracted_data)

In [25]:
texts_chunks

[Document(metadata={'producer': 'pdfTeX-1.40.17', 'creator': 'LaTeX with hyperref package', 'creationdate': '2020-07-30T00:36:57+00:00', 'author': '', 'keywords': '', 'moddate': '2020-07-30T00:36:57+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.17 (TeX Live 2016) kpathsea version 6.2.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data/2007.14874v1.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1'}, page_content='Detecting bearish and bullish markets in ﬁnancial\ntime series using hierarchical hidden Markov models\nLennart Oelschl¨ ager1∗ and Timo Adam 1,2\n1Bielefeld University, Germany\n2University of St Andrews, UK\nJuly 30, 2020\nAbstract\nFinancial markets exhibit alternating periods of rising and falling prices.\nStock traders seeking to make proﬁtable investment decisions have to account\nfor those trends, where the goal is to accurately predict switches from bullish\ntowards bearish markets and vice versa. Popular tools for modeling

In [26]:
print(len(texts_chunks))

55


In [27]:
#Download Embedding Model 
def download_embedding_model():
    embedding_model = OpenAIEmbeddings()
    return embedding_model

In [28]:
embeding_model = download_embedding_model()

In [29]:
docsearch = FAISS.from_documents(texts_chunks, embeding_model)


In [30]:
# Save FAISS index to disk — run this once after building docsearch
DB_FAISS_PATH = "vectorstore/db_faiss"
docsearch.save_local(DB_FAISS_PATH)
print(f"FAISS index saved to {DB_FAISS_PATH}")

FAISS index saved to vectorstore/db_faiss


In [31]:
# Load FAISS index from disk — run this instead of rebuilding after a kernel restart
DB_FAISS_PATH = "vectorstore/db_faiss"
docsearch = FAISS.load_local(DB_FAISS_PATH, embeding_model, allow_dangerous_deserialization=True)
print("FAISS index loaded from disk")

FAISS index loaded from disk


In [37]:
docsearch.similarity_search("What is the main idea of the document?", k=3)

[Document(id='1e6f9a11-9c72-435d-87b6-4c8fd73dfda8', metadata={'producer': 'pdfTeX-1.40.17', 'creator': 'LaTeX with hyperref package', 'creationdate': '2020-07-30T00:36:57+00:00', 'author': '', 'keywords': '', 'moddate': '2020-07-30T00:36:57+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.17 (TeX Live 2016) kpathsea version 6.2.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data/2007.14874v1.pdf', 'total_pages': 18, 'page': 14, 'page_label': '15'}, page_content='●\n●●\n●\n●\n●\n●\n●●\n●\n●\n●●\n●\n●\n●\n●\n●●\n●\n●\n●\n●\n●\n●\n●\n●●\n●\n●\n●●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●●\n●\n●\n●\n●●●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●●\n●\n●\n●\n●\n●●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●●\n●\n●●\n●\n●\n●\n●\n●\n●\n●●\n●\n●\n●\n●●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●●●\n●\n●\n●\n●\n●●\n●●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●●\n●\n●\n●\n●\n●\n●●\n●●\n●

In [33]:
prompt_template = """
Use the following pieces of information to answer the user's question 
If you don't know the answer, just say that you don't know 

Context: {context}
Question: {question}

Only return the helpful answer below and nothing else 
Helpful answer:
"""



In [34]:
PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"],
)




In [35]:
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=docsearch.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT},
)



In [36]:
qa.invoke({"query": "What is the main idea of the document?"})

{'query': 'What is the main idea of the document?',
 'result': 'The main idea of the document is to use the hierarchical extension of hidden Markov models (HMMs) to detect bearish and bullish markets in stock trading.',
 'source_documents': [Document(id='1e6f9a11-9c72-435d-87b6-4c8fd73dfda8', metadata={'producer': 'pdfTeX-1.40.17', 'creator': 'LaTeX with hyperref package', 'creationdate': '2020-07-30T00:36:57+00:00', 'author': '', 'keywords': '', 'moddate': '2020-07-30T00:36:57+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.17 (TeX Live 2016) kpathsea version 6.2.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data/2007.14874v1.pdf', 'total_pages': 18, 'page': 14, 'page_label': '15'}, page_content='●\n●●\n●\n●\n●\n●\n●●\n●\n●\n●●\n●\n●\n●\n●\n●●\n●\n●\n●\n●\n●\n●\n●\n●●\n●\n●\n●●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●●\n●\n●\n●\n●●●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●●\n●\n●\n●\n●\n●●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●\n●●\n●\n●●\n●\n●\n●\n●\n●\n●